# 02 — Data Quality Audit

## Objective

Evaluate the completeness, uniqueness, validity, consistency, and referential integrity of the insurance datasets before data cleaning and business analysis.

### Audit Scope
- Missing values and duplicate records
- Primary and foreign key integrity
- Date and chronological consistency
- Numerical range validation
- Categorical and business-rule validation
- Claims, payments, and policy consistency
- Financial reconciliation and anomaly detection

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("../data")

datasets = {
    "Customers": pd.read_csv(data_path / "customers.csv"),
    "Policies": pd.read_csv(data_path / "policies.csv"),
    "Claims": pd.read_csv(data_path / "claims.csv"),
    "Payments": pd.read_csv(data_path / "payments.csv"),
    "Risk Assessment": pd.read_csv(data_path / "risk_assessment.csv"),
    "Agents": pd.read_csv(data_path / "agents.csv"),
    "Policy Agents": pd.read_csv(data_path / "policy_agents.csv"),
    "Vehicles": pd.read_csv(data_path / "vehicles.csv")
}

print("All datasets loaded successfully!")

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows")

All datasets loaded successfully!
Customers: 10,000 rows
Policies: 15,000 rows
Claims: 25,000 rows
Payments: 15,541 rows
Risk Assessment: 10,000 rows
Agents: 250 rows
Policy Agents: 15,000 rows
Vehicles: 6,000 rows


In [2]:
missing_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Total_Rows": len(df),
        "Total_Columns": df.shape[1],
        "Missing_Values": df.isna().sum().sum(),
        "Missing_Percentage": round(
            df.isna().sum().sum() / df.size * 100, 2
        )
    }
    for name, df in datasets.items()
])

missing_summary["Status"] = np.where(
    missing_summary["Missing_Values"] == 0,
    "PASS",
    "REVIEW"
)

display(missing_summary)

,Dataset,Total_Rows,Total_Columns,Missing_Values,Missing_Percentage,Status
0,Customers,10000,6,0,0.0,PASS
1,Policies,15000,8,0,0.0,PASS
2,Claims,25000,7,0,0.0,PASS
3,Payments,15541,5,0,0.0,PASS
4,Risk Assessment,10000,4,0,0.0,PASS
5,Agents,250,4,0,0.0,PASS
6,Policy Agents,15000,2,0,0.0,PASS
7,Vehicles,6000,5,0,0.0,PASS


In [3]:
# Blank and Whitespace Value Audit

blank_results = []

for name, df in datasets.items():

    text_columns = df.select_dtypes(
        include=["object", "str", "string"]
    ).columns

    for col in text_columns:

        blank_count = (
            df[col].notna() &
            df[col].astype("string").str.strip().eq("")
        ).sum()

        blank_results.append({
            "Dataset": name,
            "Column": col,
            "Blank_Values": int(blank_count),
            "Status": "PASS" if blank_count == 0 else "REVIEW"
        })

blank_summary = pd.DataFrame(blank_results)

display(blank_summary)

print(
    "\nTotal Blank Values:",
    blank_summary["Blank_Values"].sum()
)

,Dataset,Column,Blank_Values,Status
0,Customers,Gender,0,PASS
1,Customers,Region,0,PASS
2,Customers,Join_Date,0,PASS
3,Policies,Policy_Type,0,PASS
4,Policies,Start_Date,0,PASS
5,Policies,End_Date,0,PASS
6,Policies,Policy_Status,0,PASS
7,Claims,Claim_Date,0,PASS
8,Claims,Claim_Status,0,PASS
9,Claims,Claim_Type,0,PASS



Total Blank Values: 0


In [4]:
# Duplicate Records Audit

primary_keys = {
    "Customers": ["Customer_ID"],
    "Policies": ["Policy_ID"],
    "Claims": ["Claim_ID"],
    "Payments": ["Payment_ID"],
    "Risk Assessment": ["Customer_ID"],
    "Agents": ["Agent_ID"],
    "Policy Agents": ["Policy_ID", "Agent_ID"],
    "Vehicles": ["Vehicle_ID"]
}

duplicate_results = []

for name, df in datasets.items():

    exact_duplicates = df.duplicated().sum()

    key_duplicates = df.duplicated(
        subset=primary_keys[name]
    ).sum()

    duplicate_results.append({
        "Dataset": name,
        "Exact_Duplicates": int(exact_duplicates),
        "Key_Duplicates": int(key_duplicates),
        "Status": (
            "PASS"
            if exact_duplicates == 0 and key_duplicates == 0
            else "REVIEW"
        )
    })

duplicate_summary = pd.DataFrame(duplicate_results)

display(duplicate_summary)

print(
    "\nTotal Exact Duplicates:",
    duplicate_summary["Exact_Duplicates"].sum()
)

print(
    "Total Key Duplicates:",
    duplicate_summary["Key_Duplicates"].sum()
)

,Dataset,Exact_Duplicates,Key_Duplicates,Status
0,Customers,0,0,PASS
1,Policies,0,0,PASS
2,Claims,0,0,PASS
3,Payments,0,0,PASS
4,Risk Assessment,0,0,PASS
5,Agents,0,0,PASS
6,Policy Agents,0,0,PASS
7,Vehicles,0,0,PASS



Total Exact Duplicates: 0
Total Key Duplicates: 0


In [5]:
# Primary and Foreign Key Integrity Audit

relationships = [
    ("Policies", "Customer_ID", "Customers", "Customer_ID"),
    ("Claims", "Policy_ID", "Policies", "Policy_ID"),
    ("Payments", "Claim_ID", "Claims", "Claim_ID"),
    ("Risk Assessment", "Customer_ID", "Customers", "Customer_ID"),
    ("Policy Agents", "Policy_ID", "Policies", "Policy_ID"),
    ("Policy Agents", "Agent_ID", "Agents", "Agent_ID"),
    ("Vehicles", "Customer_ID", "Customers", "Customer_ID")
]

integrity_results = []

for child, child_key, parent, parent_key in relationships:

    child_values = datasets[child][child_key]
    parent_values = datasets[parent][parent_key]

    null_keys = child_values.isna().sum()

    invalid_references = (
        child_values.notna()
        & ~child_values.isin(parent_values)
    ).sum()

    integrity_results.append({
        "Child_Table": child,
        "Foreign_Key": child_key,
        "Parent_Table": parent,
        "Null_Keys": int(null_keys),
        "Invalid_References": int(invalid_references),
        "Status": (
            "PASS"
            if null_keys == 0 and invalid_references == 0
            else "REVIEW"
        )
    })

integrity_summary = pd.DataFrame(integrity_results)

display(integrity_summary)

,Child_Table,Foreign_Key,Parent_Table,Null_Keys,Invalid_References,Status
0,Policies,Customer_ID,Customers,0,0,PASS
1,Claims,Policy_ID,Policies,0,0,PASS
2,Payments,Claim_ID,Claims,0,0,PASS
3,Risk Assessment,Customer_ID,Customers,0,0,PASS
4,Policy Agents,Policy_ID,Policies,0,0,PASS
5,Policy Agents,Agent_ID,Agents,0,0,PASS
6,Vehicles,Customer_ID,Customers,0,0,PASS


In [6]:
# Validate Date Formats and Ranges

date_columns = {
    "Customers": ["Join_Date"],
    "Policies": ["Start_Date", "End_Date"],
    "Claims": ["Claim_Date"],
    "Payments": ["Payment_Date"],
    "Risk Assessment": ["Assessment_Date"]
}

date_audit = []

for name, columns in date_columns.items():
    for col in columns:

        dates = pd.to_datetime(
            datasets[name][col],
            errors="coerce"
        )

        invalid_dates = (
            datasets[name][col].notna() & dates.isna()
        ).sum()

        date_audit.append({
            "Dataset": name,
            "Date_Column": col,
            "Earliest_Date": dates.min(),
            "Latest_Date": dates.max(),
            "Invalid_Dates": int(invalid_dates),
            "Status": "PASS" if invalid_dates == 0 else "REVIEW"
        })

date_audit_summary = pd.DataFrame(date_audit)

display(date_audit_summary)

,Dataset,Date_Column,Earliest_Date,Latest_Date,Invalid_Dates,Status
0,Customers,Join_Date,2016-01-01,2024-12-30,0,PASS
1,Policies,Start_Date,2020-01-01,2025-12-30,0,PASS
2,Policies,End_Date,2020-12-31,2028-12-29,0,PASS
3,Claims,Claim_Date,2020-01-05,2026-09-30,0,PASS
4,Payments,Payment_Date,2020-02-13,2026-12-24,0,PASS
5,Risk Assessment,Assessment_Date,2025-01-01,2026-09-29,0,PASS


In [7]:
policy_dates = datasets["Policies"].copy()

policy_dates["Start_Date"] = pd.to_datetime(
    policy_dates["Start_Date"],
    errors="coerce"
)

policy_dates["End_Date"] = pd.to_datetime(
    policy_dates["End_Date"],
    errors="coerce"
)

invalid_policy_dates = (
    policy_dates["End_Date"] <= policy_dates["Start_Date"]
)

policy_chronology_summary = pd.DataFrame({
    "Validation": ["Policy End Date > Start Date"],
    "Total_Policies": [len(policy_dates)],
    "Invalid_Records": [int(invalid_policy_dates.sum())],
    "Status": [
        "PASS" if invalid_policy_dates.sum() == 0 else "REVIEW"
    ]
})

display(policy_chronology_summary)

,Validation,Total_Policies,Invalid_Records,Status
0,Policy End Date > Start Date,15000,0,PASS


In [8]:
claims_policy = datasets["Claims"].merge(
    datasets["Policies"][
        ["Policy_ID", "Start_Date", "End_Date"]
    ],
    on="Policy_ID",
    how="left",
    validate="many_to_one"
)

date_cols = ["Claim_Date", "Start_Date", "End_Date"]

for col in date_cols:
    claims_policy[col] = pd.to_datetime(
        claims_policy[col],
        errors="coerce"
    )

claims_before_start = (
    claims_policy["Claim_Date"] < claims_policy["Start_Date"]
).sum()

claims_after_end = (
    claims_policy["Claim_Date"] > claims_policy["End_Date"]
).sum()

invalid_dates = (
    claims_policy[date_cols].isna().any(axis=1)
).sum()

claim_coverage_summary = pd.DataFrame({
    "Validation": [
        "Claims Before Policy Start",
        "Claims After Policy End",
        "Missing or Invalid Dates"
    ],
    "Invalid_Records": [
        claims_before_start,
        claims_after_end,
        invalid_dates
    ]
})

claim_coverage_summary["Status"] = np.where(
    claim_coverage_summary["Invalid_Records"] == 0,
    "PASS",
    "REVIEW"
)

display(claim_coverage_summary)

,Validation,Invalid_Records,Status
0,Claims Before Policy Start,0,PASS
1,Claims After Policy End,0,PASS
2,Missing or Invalid Dates,0,PASS


In [9]:
# Payment Date vs. Claim Date

payment_claims = datasets["Payments"].merge(
    datasets["Claims"][["Claim_ID", "Claim_Date"]],
    on="Claim_ID",
    how="left",
    validate="many_to_one"
)

payment_claims["Payment_Date"] = pd.to_datetime(
    payment_claims["Payment_Date"],
    errors="coerce"
)

payment_claims["Claim_Date"] = pd.to_datetime(
    payment_claims["Claim_Date"],
    errors="coerce"
)

payment_before_claim = (
    payment_claims["Payment_Date"] <
    payment_claims["Claim_Date"]
).sum()

invalid_dates = payment_claims[
    ["Payment_Date", "Claim_Date"]
].isna().any(axis=1).sum()

payment_date_summary = pd.DataFrame({
    "Validation": [
        "Payments Before Claim Date",
        "Missing or Invalid Dates"
    ],
    "Invalid_Records": [
        int(payment_before_claim),
        int(invalid_dates)
    ]
})

payment_date_summary["Status"] = np.where(
    payment_date_summary["Invalid_Records"] == 0,
    "PASS",
    "REVIEW"
)

display(payment_date_summary)

,Validation,Invalid_Records,Status
0,Payments Before Claim Date,0,PASS
1,Missing or Invalid Dates,0,PASS


In [10]:
# Customer Join Date vs. Policy Start Date

customer_policies = datasets["Policies"].merge(
    datasets["Customers"][["Customer_ID", "Join_Date"]],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

customer_policies["Start_Date"] = pd.to_datetime(
    customer_policies["Start_Date"],
    errors="coerce"
)

customer_policies["Join_Date"] = pd.to_datetime(
    customer_policies["Join_Date"],
    errors="coerce"
)

policies_before_join = (
    customer_policies["Start_Date"] <
    customer_policies["Join_Date"]
).sum()

invalid_dates = customer_policies[
    ["Start_Date", "Join_Date"]
].isna().any(axis=1).sum()

customer_policy_summary = pd.DataFrame({
    "Validation": [
        "Policies Before Customer Join Date",
        "Missing or Invalid Dates"
    ],
    "Invalid_Records": [
        int(policies_before_join),
        int(invalid_dates)
    ]
})

customer_policy_summary["Status"] = np.where(
    customer_policy_summary["Invalid_Records"] == 0,
    "PASS",
    "REVIEW"
)

display(customer_policy_summary)

,Validation,Invalid_Records,Status
0,Policies Before Customer Join Date,3512,REVIEW
1,Missing or Invalid Dates,0,PASS


In [11]:
# Investigate Date Inconsistencies

invalid_customer_policies = customer_policies[
    customer_policies["Start_Date"] <
    customer_policies["Join_Date"]
].copy()

invalid_customer_policies["Days_Before_Join"] = (
    invalid_customer_policies["Join_Date"]
    - invalid_customer_policies["Start_Date"]
).dt.days

print("Affected Policies:", len(invalid_customer_policies))

print(
    "Affected Customers:",
    invalid_customer_policies["Customer_ID"].nunique()
)

display(
    invalid_customer_policies[
        [
            "Policy_ID",
            "Customer_ID",
            "Join_Date",
            "Start_Date",
            "Days_Before_Join"
        ]
    ].head(10)
)

display(
    invalid_customer_policies["Days_Before_Join"]
    .describe()
    .round(2)
)

Affected Policies: 3512
Affected Customers: 2390


,Policy_ID,Customer_ID,Join_Date,Start_Date,Days_Before_Join
3,4,2278,2023-01-02,2020-09-17,837
4,5,4489,2024-02-06,2022-08-04,551
5,6,369,2024-04-27,2024-03-04,54
15,16,1645,2024-05-07,2020-07-15,1392
17,18,7094,2022-08-25,2021-08-19,371
21,22,5389,2022-01-02,2020-12-07,391
31,32,5419,2022-10-05,2020-06-10,847
33,34,4741,2024-06-14,2022-11-22,570
35,36,6918,2020-10-05,2020-02-24,224
47,48,8138,2024-12-23,2024-06-04,202


count    3512.00
mean      612.60
std       428.39
min         1.00
25%       248.75
50%       544.00
75%       915.25
max      1808.00
Name: Days_Before_Join, dtype: float64

In [12]:
# Step 2.6.7 — Validate Policy Status Against Coverage Dates

reference_date = pd.Timestamp("2026-09-30")

policy_status_audit = datasets["Policies"].copy()

policy_status_audit["Start_Date"] = pd.to_datetime(
    policy_status_audit["Start_Date"]
)

policy_status_audit["End_Date"] = pd.to_datetime(
    policy_status_audit["End_Date"]
)

# Identify potentially inconsistent policy statuses
active_expired = (
    (policy_status_audit["Policy_Status"] == "Active") &
    (policy_status_audit["End_Date"] < reference_date)
)

expired_not_ended = (
    (policy_status_audit["Policy_Status"] == "Expired") &
    (policy_status_audit["End_Date"] >= reference_date)
)

future_active = (
    (policy_status_audit["Policy_Status"] == "Active") &
    (policy_status_audit["Start_Date"] > reference_date)
)

# Create validation summary
policy_status_summary = pd.DataFrame({
    "Validation": [
        "Active Policies Past End Date",
        "Expired Policies Not Yet Ended",
        "Active Policies Starting in Future"
    ],
    "Affected_Records": [
        int(active_expired.sum()),
        int(expired_not_ended.sum()),
        int(future_active.sum())
    ]
})

policy_status_summary["Status"] = np.where(
    policy_status_summary["Affected_Records"] == 0,
    "PASS",
    "REVIEW"
)

display(policy_status_summary)

,Validation,Affected_Records,Status
0,Active Policies Past End Date,7306,REVIEW
1,Expired Policies Not Yet Ended,618,REVIEW
2,Active Policies Starting in Future,0,PASS


In [13]:
# Step 2.6.8 — Investigate Policy Status and Coverage Date Inconsistencies

# Extract active policies with expired coverage
active_expired_policies = policy_status_audit[
    active_expired
].copy()

active_expired_policies["Days_Since_Expiry"] = (
    reference_date - active_expired_policies["End_Date"]
).dt.days

# Extract expired policies whose coverage has not ended
expired_future_policies = policy_status_audit[
    expired_not_ended
].copy()

expired_future_policies["Days_Until_End"] = (
    expired_future_policies["End_Date"] - reference_date
).dt.days

# Display affected records
print("Active Policies Past End Date:", len(active_expired_policies))
print("Expired Policies Not Yet Ended:", len(expired_future_policies))

print("\nSample Active Policies Past End Date:")
display(
    active_expired_policies[
        ["Policy_ID", "Start_Date", "End_Date",
         "Policy_Status", "Days_Since_Expiry"]
    ].head(10)
)

print("\nDays Since Expiry — Summary:")
display(
    active_expired_policies["Days_Since_Expiry"]
    .describe()
    .round(2)
)

print("\nDays Until End — Summary:")
display(
    expired_future_policies["Days_Until_End"]
    .describe()
    .round(2)
)

Active Policies Past End Date: 7306
Expired Policies Not Yet Ended: 618

Sample Active Policies Past End Date:


,Policy_ID,Start_Date,End_Date,Policy_Status,Days_Since_Expiry
1,2,2025-05-08,2026-05-08,Active,145
2,3,2022-12-14,2024-12-13,Active,656
4,5,2022-08-04,2023-08-04,Active,1153
5,6,2024-03-04,2026-03-04,Active,210
6,7,2021-11-28,2022-11-28,Active,1402
8,9,2020-03-28,2021-03-28,Active,2012
10,11,2020-06-20,2021-06-20,Active,1928
12,13,2023-06-20,2024-06-19,Active,833
13,14,2024-06-12,2025-06-12,Active,475
20,21,2025-09-11,2026-09-11,Active,19



Days Since Expiry — Summary:


count    7306.00
mean      993.58
std       581.77
min         1.00
25%       494.00
50%       981.00
75%      1481.00
max      2099.00
Name: Days_Since_Expiry, dtype: float64


Days Until End — Summary:


count    618.00
mean     257.10
std      219.74
min        0.00
25%       58.00
50%      221.00
75%      392.75
max      816.00
Name: Days_Until_End, dtype: float64

In [14]:
# Step 2.6.9 — Validate Risk Assessment Dates

risk_dates = datasets["Risk Assessment"].merge(
    datasets["Customers"][["Customer_ID", "Join_Date"]],
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

risk_dates["Assessment_Date"] = pd.to_datetime(
    risk_dates["Assessment_Date"],
    errors="coerce"
)

risk_dates["Join_Date"] = pd.to_datetime(
    risk_dates["Join_Date"],
    errors="coerce"
)

# Identify chronological inconsistencies
assessment_before_join = (
    risk_dates["Assessment_Date"] < risk_dates["Join_Date"]
).sum()

assessment_after_reference = (
    risk_dates["Assessment_Date"] > reference_date
).sum()

invalid_assessment_dates = risk_dates[
    ["Assessment_Date", "Join_Date"]
].isna().any(axis=1).sum()

# Create audit summary
risk_date_summary = pd.DataFrame({
    "Validation": [
        "Assessments Before Customer Join Date",
        "Assessments After Reference Date",
        "Missing or Invalid Dates"
    ],
    "Affected_Records": [
        int(assessment_before_join),
        int(assessment_after_reference),
        int(invalid_assessment_dates)
    ]
})

risk_date_summary["Status"] = np.where(
    risk_date_summary["Affected_Records"] == 0,
    "PASS",
    "REVIEW"
)

display(risk_date_summary)

,Validation,Affected_Records,Status
0,Assessments Before Customer Join Date,0,PASS
1,Assessments After Reference Date,0,PASS
2,Missing or Invalid Dates,0,PASS


In [15]:
# Step 2.7.1 — Validate Numerical Ranges and Business Rules

numerical_rules = [
    ("Customers", "Age", 18, 100),
    ("Customers", "Annual_Income", 0, None),
    ("Policies", "Annual_Premium", 0, None),
    ("Policies", "Coverage_Amount", 0, None),
    ("Claims", "Claim_Amount", 0, None),
    ("Payments", "Payment_Amount", 0, None),
    ("Risk Assessment", "Risk_Score", 0, 100),
    ("Agents", "Experience_Years", 0, None),
    ("Vehicles", "Vehicle_Age", 0, None),
    ("Vehicles", "Insured_Value", 0, None)
]

numerical_audit = []

for name, column, min_value, max_value in numerical_rules:

    values = datasets[name][column]

    # Identify values outside the accepted range
    invalid = values < min_value

    if max_value is not None:
        invalid = invalid | (values > max_value)

    numerical_audit.append({
        "Dataset": name,
        "Column": column,
        "Minimum": values.min(),
        "Maximum": values.max(),
        "Invalid_Records": int(invalid.sum()),
        "Status": "PASS" if invalid.sum() == 0 else "REVIEW"
    })

numerical_audit_summary = pd.DataFrame(numerical_audit)

display(numerical_audit_summary)

,Dataset,Column,Minimum,Maximum,Invalid_Records,Status
0,Customers,Age,18.00,80.00,0,PASS
1,Customers,Annual_Income,180109.00,2499909.00,0,PASS
2,Policies,Annual_Premium,1501.82,222852.66,0,PASS
3,Policies,Coverage_Amount,30162.71,7180476.75,0,PASS
4,Claims,Claim_Amount,450.79,961424.75,0,PASS
5,Payments,Payment_Amount,296.07,634466.82,0,PASS
6,Risk Assessment,Risk_Score,0.00,100.00,0,PASS
7,Agents,Experience_Years,1.00,20.00,0,PASS
8,Vehicles,Vehicle_Age,0.00,15.00,0,PASS
9,Vehicles,Insured_Value,100669.00,2999982.00,0,PASS


In [16]:
# Step 2.7.2 — Validate Claim, Payment, and Coverage Amounts

# Join claims with policy coverage information
claims_financial = datasets["Claims"].merge(
    datasets["Policies"][["Policy_ID", "Coverage_Amount"]],
    on="Policy_ID",
    how="left",
    validate="many_to_one"
)

# Identify claims exceeding policy coverage
claims_above_coverage = (
    claims_financial["Claim_Amount"] >
    claims_financial["Coverage_Amount"]
).sum()

# Join payments with claim and coverage information
payments_financial = datasets["Payments"].merge(
    claims_financial[
        ["Claim_ID", "Claim_Amount", "Coverage_Amount"]
    ],
    on="Claim_ID",
    how="left",
    validate="many_to_one"
)

# Identify payments exceeding claim amounts
payments_above_claim = (
    payments_financial["Payment_Amount"] >
    payments_financial["Claim_Amount"]
).sum()

# Identify payments exceeding coverage amounts
payments_above_coverage = (
    payments_financial["Payment_Amount"] >
    payments_financial["Coverage_Amount"]
).sum()

# Create financial validation summary
financial_audit_summary = pd.DataFrame({
    "Validation": [
        "Claims Exceeding Policy Coverage",
        "Payments Exceeding Claim Amount",
        "Payments Exceeding Policy Coverage"
    ],
    "Affected_Records": [
        int(claims_above_coverage),
        int(payments_above_claim),
        int(payments_above_coverage)
    ]
})

financial_audit_summary["Status"] = np.where(
    financial_audit_summary["Affected_Records"] == 0,
    "PASS",
    "REVIEW"
)

display(financial_audit_summary)

,Validation,Affected_Records,Status
0,Claims Exceeding Policy Coverage,143,REVIEW
1,Payments Exceeding Claim Amount,0,PASS
2,Payments Exceeding Policy Coverage,34,REVIEW


In [17]:
# Step 2.7.3 — Investigate Claims and Payments Exceeding Policy Coverage

# Identify claims exceeding coverage
excess_claims = claims_financial[
    claims_financial["Claim_Amount"] >
    claims_financial["Coverage_Amount"]
].copy()

# Calculate excess claim amounts
excess_claims["Excess_Amount"] = (
    excess_claims["Claim_Amount"]
    - excess_claims["Coverage_Amount"]
)

# Identify payments exceeding coverage
excess_payments = payments_financial[
    payments_financial["Payment_Amount"] >
    payments_financial["Coverage_Amount"]
].copy()

# Calculate excess payment amounts
excess_payments["Excess_Amount"] = (
    excess_payments["Payment_Amount"]
    - excess_payments["Coverage_Amount"]
)

print("Claims Exceeding Coverage:", len(excess_claims))
print("Payments Exceeding Coverage:", len(excess_payments))

print("\nSample Claims Exceeding Coverage:")
display(
    excess_claims[
        ["Claim_ID", "Policy_ID", "Claim_Amount",
         "Coverage_Amount", "Excess_Amount"]
    ].head(10)
)

print("\nExcess Claim Amount Statistics:")
display(excess_claims["Excess_Amount"].describe().round(2))

print("\nExcess Payment Amount Statistics:")
display(excess_payments["Excess_Amount"].describe().round(2))

Claims Exceeding Coverage: 143
Payments Exceeding Coverage: 34

Sample Claims Exceeding Coverage:


,Claim_ID,Policy_ID,Claim_Amount,Coverage_Amount,Excess_Amount
34,35,12234,282614.03,217395.41,65218.62
136,137,13149,261068.92,204863.75,56205.17
320,321,10393,167596.79,128920.61,38676.18
508,509,7813,107911.53,83008.87,24902.66
586,587,13823,413184.56,403802.39,9382.17
692,693,1292,91400.35,70307.96,21092.39
1235,1236,467,124620.20,114330.16,10290.04
1292,1293,4574,127186.24,97835.57,29350.67
1406,1407,13135,324043.21,249264.01,74779.20
1415,1416,13837,73702.75,56694.42,17008.33



Excess Claim Amount Statistics:


count       143.00
mean      29832.12
std       21050.37
min         170.68
25%       14623.65
50%       25428.91
75%       42168.17
max      117249.86
Name: Excess_Amount, dtype: float64


Excess Payment Amount Statistics:


count       34.00
mean     18913.62
std      16460.56
min        929.87
25%       3995.15
50%      16026.37
75%      28145.74
max      73490.17
Name: Excess_Amount, dtype: float64

In [18]:
# Step 2.7.4 — Validate Claim Status Against Payment Records

# Identify claims with payment records
paid_claim_ids = datasets["Payments"]["Claim_ID"]

# Create a temporary claims audit table
claims_payment_audit = datasets["Claims"].copy()

claims_payment_audit["Has_Payment"] = (
    claims_payment_audit["Claim_ID"].isin(paid_claim_ids)
)

# Summarize claim statuses and payment availability
payment_status_summary = pd.crosstab(
    claims_payment_audit["Claim_Status"],
    claims_payment_audit["Has_Payment"]
)

payment_status_summary = payment_status_summary.rename(
    columns={
        False: "Without_Payment",
        True: "With_Payment"
    }
)

display(payment_status_summary)

# Identify potential business-rule inconsistencies
settled_without_payment = (
    (claims_payment_audit["Claim_Status"] == "Settled") &
    (~claims_payment_audit["Has_Payment"])
).sum()

rejected_with_payment = (
    (claims_payment_audit["Claim_Status"] == "Rejected") &
    (claims_payment_audit["Has_Payment"])
).sum()

pending_with_payment = (
    (claims_payment_audit["Claim_Status"] == "Pending") &
    (claims_payment_audit["Has_Payment"])
).sum()

# Create validation summary
claim_payment_summary = pd.DataFrame({
    "Validation": [
        "Settled Claims Without Payment",
        "Rejected Claims With Payment",
        "Pending Claims With Payment"
    ],
    "Affected_Records": [
        int(settled_without_payment),
        int(rejected_with_payment),
        int(pending_with_payment)
    ]
})

claim_payment_summary["Status"] = np.where(
    claim_payment_summary["Affected_Records"] == 0,
    "PASS",
    "REVIEW"
)

display(claim_payment_summary)

Has_Payment,Without_Payment,With_Payment
Claim_Status,,
Approved,396,5704
Pending,4075,0
Rejected,4256,0
Settled,732,9837


,Validation,Affected_Records,Status
0,Settled Claims Without Payment,732,REVIEW
1,Rejected Claims With Payment,0,PASS
2,Pending Claims With Payment,0,PASS


In [19]:
# Step 2.7.5 — Investigate Settled Claims Without Payment Records

# Extract settled claims without payments
settled_unpaid = claims_payment_audit[
    (claims_payment_audit["Claim_Status"] == "Settled") &
    (~claims_payment_audit["Has_Payment"])
].copy()

# Calculate affected claim amount
total_unpaid_claim_amount = settled_unpaid["Claim_Amount"].sum()

print("Settled Claims Without Payments:", len(settled_unpaid))

print(
    "Total Claim Amount:",
    round(total_unpaid_claim_amount, 2)
)

print("\nClaim Amount Statistics:")
display(
    settled_unpaid["Claim_Amount"]
    .describe()
    .round(2)
)

print("\nAffected Claims by Claim Type:")
display(
    settled_unpaid["Claim_Type"]
    .value_counts()
)

print("\nAffected Claims by Fraud Flag:")
display(
    settled_unpaid["Fraud_Flag"]
    .value_counts(dropna=False)
)

print("\nSample Affected Claims:")
display(
    settled_unpaid[
        [
            "Claim_ID",
            "Policy_ID",
            "Claim_Date",
            "Claim_Amount",
            "Claim_Status",
            "Fraud_Flag",
            "Claim_Type"
        ]
    ].head(10)
)

Settled Claims Without Payments: 732
Total Claim Amount: 31687328.9

Claim Amount Statistics:


count       732.00
mean      43288.70
std       44191.82
min        1988.79
25%       17713.95
50%       30803.97
75%       52579.27
max      550416.23
Name: Claim_Amount, dtype: float64


Affected Claims by Claim Type:


Claim_Type
Auto      295
Health    228
Home      116
Life       93
Name: count, dtype: int64


Affected Claims by Fraud Flag:


Fraud_Flag
0    700
1     32
Name: count, dtype: int64


Sample Affected Claims:


,Claim_ID,Policy_ID,Claim_Date,Claim_Amount,Claim_Status,Fraud_Flag,Claim_Type
0,1,2178,2023-12-30,24036.53,Settled,0,Health
18,19,8396,2024-01-27,13964.43,Settled,0,Life
63,64,2087,2025-11-06,19898.75,Settled,0,Life
114,115,6977,2020-09-15,38256.06,Settled,0,Auto
120,121,8685,2022-08-20,18135.33,Settled,0,Health
132,133,8961,2024-05-04,7759.77,Settled,0,Auto
147,148,3951,2021-06-02,5695.98,Settled,0,Auto
190,191,14573,2024-08-23,34273.72,Settled,0,Home
245,246,10983,2026-02-03,14761.71,Settled,0,Home
313,314,5615,2024-12-22,27258.64,Settled,0,Life


In [20]:
# Step 2.7.6 — Investigate Approved Claims With Payment Records

# Select approved claims that have payments
approved_paid = claims_payment_audit[
    (claims_payment_audit["Claim_Status"] == "Approved") &
    (claims_payment_audit["Has_Payment"])
].copy()

# Calculate total payments per claim
payment_totals = (
    datasets["Payments"]
    .groupby("Claim_ID", as_index=False)
    .agg(Total_Payment=("Payment_Amount", "sum"))
)

# Attach payment totals to approved claims
approved_paid = approved_paid.merge(
    payment_totals,
    on="Claim_ID",
    how="left",
    validate="one_to_one"
)

# Calculate payment-to-claim ratio
approved_paid["Payment_Ratio"] = (
    approved_paid["Total_Payment"] /
    approved_paid["Claim_Amount"]
)

# Summarize payment relationships
payment_relationship = pd.DataFrame({
    "Payment_Category": [
        "Payment Below Claim Amount",
        "Payment Equal to Claim Amount",
        "Payment Above Claim Amount"
    ],
    "Claim_Count": [
        int((approved_paid["Total_Payment"] <
             approved_paid["Claim_Amount"]).sum()),

        int(np.isclose(
            approved_paid["Total_Payment"],
            approved_paid["Claim_Amount"],
            atol=0.01,
            rtol=0
        ).sum()),

        int((approved_paid["Total_Payment"] >
             approved_paid["Claim_Amount"] + 0.01).sum())
    ]
})

print("Approved Claims With Payments:", len(approved_paid))

print("\nPayment Relationship Summary:")
display(payment_relationship)

print("\nPayment Ratio Statistics:")
display(
    approved_paid["Payment_Ratio"]
    .describe()
    .round(4)
)

print("\nSample Approved Claims With Payments:")
display(
    approved_paid[
        [
            "Claim_ID",
            "Claim_Amount",
            "Total_Payment",
            "Payment_Ratio"
        ]
    ].head(10)
)

Approved Claims With Payments: 5704

Payment Relationship Summary:


,Payment_Category,Claim_Count
0,Payment Below Claim Amount,5704
1,Payment Equal to Claim Amount,0
2,Payment Above Claim Amount,0



Payment Ratio Statistics:


count    5704.0000
mean        0.7745
std         0.1280
min         0.5501
25%         0.6640
50%         0.7743
75%         0.8843
max         0.9998
Name: Payment_Ratio, dtype: float64


Sample Approved Claims With Payments:


,Claim_ID,Claim_Amount,Total_Payment,Payment_Ratio
0,3,13574.30,12619.67,0.929674
1,4,26658.91,21102.05,0.791557
2,16,54970.72,42368.78,0.770752
3,26,37117.95,30922.47,0.833087
4,28,28130.79,16221.21,0.576635
5,30,12058.23,7945.41,0.658920
6,39,51449.13,37802.01,0.734745
7,46,53104.13,34262.21,0.645189
8,51,6826.08,6643.40,0.973238
9,52,69860.68,63008.87,0.901922


In [21]:
# Step 2.7.7 — Validate Claim Type Against Policy Type

# Join claims with their corresponding policy types
claim_type_audit = datasets["Claims"].merge(
    datasets["Policies"][["Policy_ID", "Policy_Type"]],
    on="Policy_ID",
    how="left",
    validate="many_to_one"
)

# Compare claim type and policy type
claim_type_audit["Type_Mismatch"] = (
    claim_type_audit["Claim_Type"].str.strip().str.lower()
    !=
    claim_type_audit["Policy_Type"].str.strip().str.lower()
)

# Calculate mismatch statistics
total_claims = len(claim_type_audit)

mismatched_claims = int(
    claim_type_audit["Type_Mismatch"].sum()
)

mismatch_percentage = (
    mismatched_claims / total_claims * 100
)

print("Total Claims:", total_claims)
print("Mismatched Claim Types:", mismatched_claims)
print("Mismatch Percentage:", round(mismatch_percentage, 2), "%")

print("\nClaim Type vs Policy Type:")
display(
    pd.crosstab(
        claim_type_audit["Claim_Type"],
        claim_type_audit["Policy_Type"]
    )
)

print("\nSample Mismatched Records:")
display(
    claim_type_audit.loc[
        claim_type_audit["Type_Mismatch"],
        ["Claim_ID", "Policy_ID", "Claim_Type", "Policy_Type"]
    ].head(10)
)

Total Claims: 25000
Mismatched Claim Types: 0
Mismatch Percentage: 0.0 %

Claim Type vs Policy Type:


Policy_Type,Auto,Health,Home,Life
Claim_Type,,,,
Auto,9832,0,0,0
Health,0,7784,0,0
Home,0,0,4458,0
Life,0,0,0,2926



Sample Mismatched Records:


,Claim_ID,Policy_ID,Claim_Type,Policy_Type


In [22]:
# Step 2.7.8 — Examine Risk Score Distribution by Risk Category

risk_audit = datasets["Risk Assessment"].copy()

# Summarize risk scores for each category
risk_category_summary = (
    risk_audit
    .groupby("Risk_Category")
    .agg(
        Customer_Count=("Customer_ID", "count"),
        Minimum_Score=("Risk_Score", "min"),
        Maximum_Score=("Risk_Score", "max"),
        Average_Score=("Risk_Score", "mean"),
        Median_Score=("Risk_Score", "median")
    )
    .round(2)
    .reset_index()
    .sort_values("Minimum_Score")
)

print("Risk Score Distribution by Category:")
display(risk_category_summary)

# Inspect risk score frequency
print("\nRisk Category Distribution:")
display(
    risk_audit["Risk_Category"]
    .value_counts()
)

# Check whether risk scores are valid
print("\nRisk Scores Outside 0–100:")
print(
    (~risk_audit["Risk_Score"].between(0, 100)).sum()
)

Risk Score Distribution by Category:


,Risk_Category,Customer_Count,Minimum_Score,Maximum_Score,Average_Score,Median_Score
1,Low,2348,0,34,22.95,25.0
2,Medium,6132,35,69,51.36,51.0
0,High,1520,70,100,79.73,78.0



Risk Category Distribution:


Risk_Category
Medium    6132
Low       2348
High      1520
Name: count, dtype: int64


Risk Scores Outside 0–100:
0


In [23]:
# Step 2.7.9 — Validate Risk Category Classification Rules

# Derive risk categories from numerical scores
risk_audit["Expected_Risk_Category"] = np.select(
    [
        risk_audit["Risk_Score"].between(0, 34),
        risk_audit["Risk_Score"].between(35, 69),
        risk_audit["Risk_Score"].between(70, 100)
    ],
    [
        "Low",
        "Medium",
        "High"
    ],
    default="Invalid"
)

# Compare recorded and expected categories
risk_audit["Category_Mismatch"] = (
    risk_audit["Risk_Category"] !=
    risk_audit["Expected_Risk_Category"]
)

# Summarize validation results
total_records = len(risk_audit)
mismatch_count = int(risk_audit["Category_Mismatch"].sum())

print("Total Risk Assessments:", total_records)
print("Category Mismatches:", mismatch_count)

print(
    "Classification Accuracy:",
    round((total_records - mismatch_count) / total_records * 100, 2),
    "%"
)

print("\nRisk Category Validation Summary:")

display(
    pd.crosstab(
        risk_audit["Risk_Category"],
        risk_audit["Expected_Risk_Category"]
    )
)

print("\nSample Mismatched Records:")

display(
    risk_audit.loc[
        risk_audit["Category_Mismatch"],
        [
            "Customer_ID",
            "Risk_Score",
            "Risk_Category",
            "Expected_Risk_Category"
        ]
    ].head(10)
)

Total Risk Assessments: 10000
Category Mismatches: 0
Classification Accuracy: 100.0 %

Risk Category Validation Summary:


Expected_Risk_Category,High,Low,Medium
Risk_Category,,,
High,1520,0,0
Low,0,2348,0
Medium,0,0,6132



Sample Mismatched Records:


,Customer_ID,Risk_Score,Risk_Category,Expected_Risk_Category


In [24]:
# Step 2.7.10 — Validate Fraud Flag Values and Distribution

fraud_audit = datasets["Claims"].copy()

# Check for invalid fraud flag values
invalid_fraud_flags = (
    ~fraud_audit["Fraud_Flag"].isin([0, 1])
).sum()

# Summarize fraud flag distribution
fraud_distribution = (
    fraud_audit["Fraud_Flag"]
    .value_counts(dropna=False)
    .rename_axis("Fraud_Flag")
    .reset_index(name="Claim_Count")
)

fraud_distribution["Percentage"] = (
    fraud_distribution["Claim_Count"]
    / len(fraud_audit) * 100
).round(2)

print("Total Claims:", len(fraud_audit))
print("Invalid Fraud Flag Values:", invalid_fraud_flags)

print("\nFraud Flag Distribution:")
display(fraud_distribution)

print("\nFraud Flag vs Claim Status:")
display(
    pd.crosstab(
        fraud_audit["Claim_Status"],
        fraud_audit["Fraud_Flag"]
    )
)

Total Claims: 25000
Invalid Fraud Flag Values: 0

Fraud Flag Distribution:


,Fraud_Flag,Claim_Count,Percentage
0,0,24079,96.32
1,1,921,3.68



Fraud Flag vs Claim Status:


Fraud_Flag,0,1
Claim_Status,,
Approved,5870,230
Pending,3926,149
Rejected,4093,163
Settled,10190,379


In [25]:
# Step 2.7.11 — Validate Total Payments Against Claim Amounts

# Aggregate payments at claim level
claim_payment_totals = (
    datasets["Payments"]
    .groupby("Claim_ID", as_index=False)
    .agg(
        Payment_Count=("Payment_ID", "count"),
        Total_Payment=("Payment_Amount", "sum")
    )
)

# Join aggregated payments with claim details
payment_reconciliation = datasets["Claims"].merge(
    claim_payment_totals,
    on="Claim_ID",
    how="left",
    validate="one_to_one"
)

# Claims without payments should have zero recorded payment amount
payment_reconciliation["Payment_Count"] = (
    payment_reconciliation["Payment_Count"].fillna(0).astype(int)
)

payment_reconciliation["Total_Payment"] = (
    payment_reconciliation["Total_Payment"].fillna(0)
)

# Identify total payments exceeding claim amounts
payment_reconciliation["Excess_Payment"] = (
    payment_reconciliation["Total_Payment"]
    - payment_reconciliation["Claim_Amount"]
)

overpaid_claims = payment_reconciliation[
    payment_reconciliation["Excess_Payment"] > 0.01
]

print("Total Claims:", len(payment_reconciliation))

print(
    "Claims With Multiple Payments:",
    (payment_reconciliation["Payment_Count"] > 1).sum()
)

print(
    "Claims With Total Payments Exceeding Claim Amount:",
    len(overpaid_claims)
)

print("\nPayment Count Distribution:")
display(
    payment_reconciliation["Payment_Count"]
    .value_counts()
    .sort_index()
)

print("\nSample Claims With Excess Payments:")
display(
    overpaid_claims[
        [
            "Claim_ID",
            "Claim_Status",
            "Claim_Amount",
            "Payment_Count",
            "Total_Payment",
            "Excess_Payment"
        ]
    ].head(10)
)

Total Claims: 25000
Claims With Multiple Payments: 0
Claims With Total Payments Exceeding Claim Amount: 0

Payment Count Distribution:


Payment_Count
0     9459
1    15541
Name: count, dtype: int64


Sample Claims With Excess Payments:


,Claim_ID,Claim_Status,Claim_Amount,Payment_Count,Total_Payment,Excess_Payment


In [26]:
# Step 2.7.12 — Validate Auto Policy Customers Against Vehicle Records

# Extract Auto insurance policies
auto_policies = datasets["Policies"][
    datasets["Policies"]["Policy_Type"] == "Auto"
].copy()

# Identify customers with vehicle records
vehicle_customer_ids = set(
    datasets["Vehicles"]["Customer_ID"]
)

# Check whether Auto policyholders have vehicle records
auto_policies["Has_Vehicle_Record"] = (
    auto_policies["Customer_ID"].isin(vehicle_customer_ids)
)

# Summarize customer-level vehicle availability
auto_vehicle_summary = pd.DataFrame({
    "Validation": [
        "Total Auto Policies",
        "Auto Policies With Vehicle Records",
        "Auto Policies Without Vehicle Records",
        "Unique Auto Policyholders",
        "Auto Policyholders Without Vehicle Records"
    ],
    "Record_Count": [
        len(auto_policies),
        int(auto_policies["Has_Vehicle_Record"].sum()),
        int((~auto_policies["Has_Vehicle_Record"]).sum()),
        auto_policies["Customer_ID"].nunique(),
        auto_policies.loc[
            ~auto_policies["Has_Vehicle_Record"],
            "Customer_ID"
        ].nunique()
    ]
})

print("Auto Policy and Vehicle Record Summary:")
display(auto_vehicle_summary)

print("\nSample Auto Policies Without Vehicle Records:")
display(
    auto_policies.loc[
        ~auto_policies["Has_Vehicle_Record"],
        [
            "Policy_ID",
            "Customer_ID",
            "Policy_Type",
            "Policy_Status"
        ]
    ].head(10)
)

Auto Policy and Vehicle Record Summary:


,Validation,Record_Count
0,Total Auto Policies,5953
1,Auto Policies With Vehicle Records,3564
2,Auto Policies Without Vehicle Records,2389
3,Unique Auto Policyholders,4474
4,Auto Policyholders Without Vehicle Records,1793



Sample Auto Policies Without Vehicle Records:


,Policy_ID,Customer_ID,Policy_Type,Policy_Status
16,17,398,Auto,Expired
19,20,5082,Auto,Cancelled
27,28,6967,Auto,Expired
35,36,6918,Auto,Expired
50,51,3552,Auto,Cancelled
59,60,6919,Auto,Expired
71,72,2874,Auto,Active
73,74,8532,Auto,Active
86,87,5859,Auto,Expired
88,89,4740,Auto,Active


In [27]:
# Step 2.7.13 — Validate Policy-Agent Assignment Consistency

# Count agent assignments for each policy
policy_agent_counts = (
    datasets["Policy Agents"]
    .groupby("Policy_ID")
    .agg(
        Agent_Count=("Agent_ID", "nunique"),
        Assignment_Count=("Agent_ID", "size")
    )
    .reset_index()
)

# Include all policies, even those without agent assignments
policy_agent_audit = datasets["Policies"][
    ["Policy_ID"]
].merge(
    policy_agent_counts,
    on="Policy_ID",
    how="left",
    validate="one_to_one"
)

policy_agent_audit[
    ["Agent_Count", "Assignment_Count"]
] = policy_agent_audit[
    ["Agent_Count", "Assignment_Count"]
].fillna(0).astype(int)

# Calculate assignment statistics
policies_without_agents = (
    policy_agent_audit["Agent_Count"] == 0
).sum()

policies_with_multiple_agents = (
    policy_agent_audit["Agent_Count"] > 1
).sum()

duplicate_assignments = (
    policy_agent_audit["Assignment_Count"]
    > policy_agent_audit["Agent_Count"]
).sum()

# Display audit summary
assignment_summary = pd.DataFrame({
    "Validation": [
        "Total Policies",
        "Policies Without Agents",
        "Policies With Multiple Agents",
        "Policies With Duplicate Assignments"
    ],
    "Record_Count": [
        len(policy_agent_audit),
        int(policies_without_agents),
        int(policies_with_multiple_agents),
        int(duplicate_assignments)
    ]
})

print("Policy-Agent Assignment Summary:")
display(assignment_summary)

print("\nAgent Assignment Count Distribution:")
display(
    policy_agent_audit["Agent_Count"]
    .value_counts()
    .sort_index()
)

Policy-Agent Assignment Summary:


,Validation,Record_Count
0,Total Policies,15000
1,Policies Without Agents,0
2,Policies With Multiple Agents,0
3,Policies With Duplicate Assignments,0



Agent Assignment Count Distribution:


Agent_Count
1    15000
Name: count, dtype: int64

In [28]:
# Step 2.7.14 — Validate Agent and Policyholder Region Consistency

# Combine policy-agent assignments with customer and agent regions
region_audit = (
    datasets["Policy Agents"]
    .merge(
        datasets["Policies"][["Policy_ID", "Customer_ID"]],
        on="Policy_ID",
        how="left",
        validate="one_to_one"
    )
    .merge(
        datasets["Customers"][["Customer_ID", "Region"]],
        on="Customer_ID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        datasets["Agents"][["Agent_ID", "Region"]],
        on="Agent_ID",
        how="left",
        validate="many_to_one",
        suffixes=("_Customer", "_Agent")
    )
)

# Compare customer and agent regions
region_audit["Region_Mismatch"] = (
    region_audit["Region_Customer"].str.strip().str.lower()
    != region_audit["Region_Agent"].str.strip().str.lower()
)

# Calculate summary statistics
total_assignments = len(region_audit)
mismatched_regions = int(region_audit["Region_Mismatch"].sum())

print("Total Policy-Agent Assignments:", total_assignments)
print("Region Mismatches:", mismatched_regions)

print(
    "Mismatch Percentage:",
    round(mismatched_regions / total_assignments * 100, 2),
    "%"
)

print("\nCustomer Region vs Agent Region:")
display(
    pd.crosstab(
        region_audit["Region_Customer"],
        region_audit["Region_Agent"]
    )
)

print("\nSample Cross-Region Assignments:")
display(
    region_audit.loc[
        region_audit["Region_Mismatch"],
        [
            "Policy_ID",
            "Customer_ID",
            "Agent_ID",
            "Region_Customer",
            "Region_Agent"
        ]
    ].head(10)
)

Total Policy-Agent Assignments: 15000
Region Mismatches: 11976
Mismatch Percentage: 79.84 %

Customer Region vs Agent Region:


Region_Agent,Central,East,North,South,West
Region_Customer,,,,,
Central,580,613,712,581,520
East,512,652,688,605,537
North,564,675,705,574,501
South,558,610,701,572,492
West,576,709,690,558,515



Sample Cross-Region Assignments:


,Policy_ID,Customer_ID,Agent_ID,Region_Customer,Region_Agent
0,1,9801,65,Central,West
1,2,2896,209,North,East
3,4,2278,90,North,West
4,5,4489,88,Central,North
5,6,369,156,South,Central
6,7,4360,108,North,East
7,8,6748,42,East,South
8,9,8802,67,West,Central
9,10,9508,176,Central,East
10,11,9806,117,Central,South


In [29]:
# Step 2.7.15 — Investigate Vehicle Age and Insured Value

vehicle_audit = datasets["Vehicles"].copy()

# Summarize vehicle age and insured value
print("Vehicle Numerical Summary:")
display(
    vehicle_audit[
        ["Vehicle_Age", "Insured_Value"]
    ].describe().round(2)
)

# Calculate IQR-based outlier boundaries
q1 = vehicle_audit["Insured_Value"].quantile(0.25)
q3 = vehicle_audit["Insured_Value"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Identify statistical outliers
vehicle_audit["Value_Outlier"] = (
    (vehicle_audit["Insured_Value"] < lower_bound) |
    (vehicle_audit["Insured_Value"] > upper_bound)
)

print("\nInsured Value Outlier Summary:")
print("Lower Bound:", round(lower_bound, 2))
print("Upper Bound:", round(upper_bound, 2))
print("Outlier Records:", vehicle_audit["Value_Outlier"].sum())

print("\nInsured Value by Vehicle Age:")
display(
    vehicle_audit.groupby("Vehicle_Age").agg(
        Vehicle_Count=("Vehicle_ID", "count"),
        Average_Insured_Value=("Insured_Value", "mean"),
        Minimum_Insured_Value=("Insured_Value", "min"),
        Maximum_Insured_Value=("Insured_Value", "max")
    ).round(2)
)

Vehicle Numerical Summary:


,Vehicle_Age,Insured_Value
count,6000.00,6000.00
mean,7.50,1534548.28
std,4.59,837402.55
min,0.00,100669.00
25%,4.00,817674.25
50%,7.00,1526316.50
75%,11.00,2264254.75
max,15.00,2999982.00



Insured Value Outlier Summary:
Lower Bound: -1352196.5
Upper Bound: 4434125.5
Outlier Records: 0

Insured Value by Vehicle Age:


,Vehicle_Count,Average_Insured_Value,Minimum_Insured_Value,Maximum_Insured_Value
Vehicle_Age,,,,
0,334,1565106.14,100669,2975286
1,411,1557230.42,106944,2999982
2,384,1535505.42,140347,2984581
3,366,1564504.67,100985,2991616
4,378,1503757.89,108236,2995294
5,365,1516115.36,102820,2997974
6,392,1469197.24,104618,2984796
7,386,1543067.21,110862,2997043
8,349,1611281.46,120273,2998219


In [30]:
# Step 2.7.16 — Validate Policy Premium-to-Coverage Ratios

premium_audit = datasets["Policies"].copy()

# Calculate annual premium as a percentage of coverage
premium_audit["Premium_Coverage_Ratio"] = (
    premium_audit["Annual_Premium"]
    / premium_audit["Coverage_Amount"]
) * 100

# Summarize the ratio distribution
print("Premium-to-Coverage Ratio Statistics (%):")
display(
    premium_audit["Premium_Coverage_Ratio"]
    .describe()
    .round(2)
)

# Examine ratios by policy type
print("\nPremium-to-Coverage Ratio by Policy Type:")
display(
    premium_audit.groupby("Policy_Type").agg(
        Policy_Count=("Policy_ID", "count"),
        Average_Ratio=("Premium_Coverage_Ratio", "mean"),
        Minimum_Ratio=("Premium_Coverage_Ratio", "min"),
        Maximum_Ratio=("Premium_Coverage_Ratio", "max")
    ).round(2)
)

# Identify statistical outliers using IQR
q1 = premium_audit["Premium_Coverage_Ratio"].quantile(0.25)
q3 = premium_audit["Premium_Coverage_Ratio"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

premium_outliers = premium_audit[
    (premium_audit["Premium_Coverage_Ratio"] < lower_bound) |
    (premium_audit["Premium_Coverage_Ratio"] > upper_bound)
]

print("\nPremium Ratio Outlier Summary:")
print("Lower Bound:", round(lower_bound, 2), "%")
print("Upper Bound:", round(upper_bound, 2), "%")
print("Outlier Policies:", len(premium_outliers))

print("\nSample Outlier Policies:")
display(
    premium_outliers[
        [
            "Policy_ID",
            "Policy_Type",
            "Annual_Premium",
            "Coverage_Amount",
            "Premium_Coverage_Ratio"
        ]
    ].head(10)
)

Premium-to-Coverage Ratio Statistics (%):


count    15000.00
mean         3.55
std          1.62
min          1.82
25%          2.26
50%          2.99
75%          4.42
max          8.33
Name: Premium_Coverage_Ratio, dtype: float64


Premium-to-Coverage Ratio by Policy Type:


,Policy_Count,Average_Ratio,Minimum_Ratio,Maximum_Ratio
Policy_Type,,,,
Auto,5953,3.55,1.82,8.33
Health,4560,3.50,1.82,8.33
Home,2691,3.57,1.82,8.33
Life,1796,3.60,1.82,8.27



Premium Ratio Outlier Summary:
Lower Bound: -0.97 %
Upper Bound: 7.66 %
Outlier Policies: 380

Sample Outlier Policies:


,Policy_ID,Policy_Type,Annual_Premium,Coverage_Amount,Premium_Coverage_Ratio
17,18,Health,23143.24,295575.56,7.829890
58,59,Auto,23323.72,300138.26,7.770992
187,188,Auto,35863.12,445184.49,8.055788
226,227,Home,33138.53,422037.57,7.852033
287,288,Auto,17933.85,216346.32,8.289418
301,302,Auto,20263.16,243189.39,8.332255
399,400,Home,30873.59,400793.97,7.703107
488,489,Auto,40644.15,528442.37,7.691312
496,497,Auto,41154.62,533524.66,7.713724
570,571,Auto,5452.06,68226.26,7.991146


### Premium-to-Coverage Ratio — Key Findings

- The average annual premium-to-coverage ratio is **3.55%**, with a median of **2.99%**.
- **380 policies (2.53%)** exceed the IQR upper boundary of approximately **7.66%**.
- Average premium ratios are similar across the four insurance types.
- These observations represent **statistical outliers**, not confirmed pricing errors.
- **Decision:** Retain all policies and investigate unusual premium ratios during further analysis.

In [31]:
# Step 2.7.17 — Detect Claim Amount Outliers by Insurance Type

claim_outlier_audit = datasets["Claims"].copy()

# Calculate IQR boundaries separately for each claim type
claim_bounds = (
    claim_outlier_audit
    .groupby("Claim_Type")["Claim_Amount"]
    .agg(
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

claim_bounds["IQR"] = (
    claim_bounds["Q3"] - claim_bounds["Q1"]
)

claim_bounds["Lower_Bound"] = (
    claim_bounds["Q1"] - 1.5 * claim_bounds["IQR"]
)

claim_bounds["Upper_Bound"] = (
    claim_bounds["Q3"] + 1.5 * claim_bounds["IQR"]
)

# Attach thresholds to individual claims
claim_outlier_audit = claim_outlier_audit.merge(
    claim_bounds[
        ["Claim_Type", "Lower_Bound", "Upper_Bound"]
    ],
    on="Claim_Type",
    how="left",
    validate="many_to_one"
)

# Flag statistical outliers
claim_outlier_audit["Is_Outlier"] = (
    (claim_outlier_audit["Claim_Amount"] <
     claim_outlier_audit["Lower_Bound"]) |
    (claim_outlier_audit["Claim_Amount"] >
     claim_outlier_audit["Upper_Bound"])
)

# Summarize results by claim type
claim_outlier_summary = (
    claim_outlier_audit
    .groupby("Claim_Type")
    .agg(
        Total_Claims=("Claim_ID", "count"),
        Average_Claim=("Claim_Amount", "mean"),
        Maximum_Claim=("Claim_Amount", "max"),
        Outlier_Count=("Is_Outlier", "sum")
    )
    .reset_index()
)

claim_outlier_summary["Outlier_Percentage"] = (
    claim_outlier_summary["Outlier_Count"]
    / claim_outlier_summary["Total_Claims"] * 100
)

print("Claim Amount Outlier Summary:")
display(claim_outlier_summary.round(2))

print("\nIQR Boundaries by Claim Type:")
display(claim_bounds.round(2))

print("\nTotal Outlier Claims:")
print(int(claim_outlier_audit["Is_Outlier"].sum()))

print("\nSample Outlier Claims:")
display(
    claim_outlier_audit.loc[
        claim_outlier_audit["Is_Outlier"],
        [
            "Claim_ID",
            "Claim_Type",
            "Claim_Amount",
            "Claim_Status",
            "Fraud_Flag"
        ]
    ].head(10)
)

Claim Amount Outlier Summary:


,Claim_Type,Total_Claims,Average_Claim,Maximum_Claim,Outlier_Count,Outlier_Percentage
0,Auto,9832,42459.97,597909.92,657,6.68
1,Health,7784,42411.00,961424.75,462,5.94
2,Home,4458,41974.46,487171.57,281,6.30
3,Life,2926,40647.21,393452.56,185,6.32



IQR Boundaries by Claim Type:


,Claim_Type,Q1,Q3,IQR,Lower_Bound,Upper_Bound
0,Auto,17145.52,52608.82,35463.30,-36049.44,105803.77
1,Health,17037.74,53154.41,36116.67,-37137.27,107329.41
2,Home,16536.71,52877.86,36341.15,-37975.01,107389.59
3,Life,16070.70,50721.96,34651.25,-35906.18,102698.83



Total Outlier Claims:
1585

Sample Outlier Claims:


,Claim_ID,Claim_Type,Claim_Amount,Claim_Status,Fraud_Flag
24,25,Home,117338.46,Settled,0
34,35,Auto,282614.03,Rejected,0
91,92,Health,115283.36,Settled,1
99,100,Home,199822.58,Settled,0
131,132,Health,127264.22,Settled,0
136,137,Home,261068.92,Settled,0
196,197,Auto,124711.80,Approved,0
219,220,Home,120603.05,Settled,0
253,254,Auto,117271.97,Approved,0
260,261,Auto,114317.40,Settled,0


### Claim Amount Outlier Analysis — Key Findings

- **1,585 claims (6.34%)** were identified as statistical outliers using the IQR method within each insurance type.
- Auto insurance recorded the highest outlier count (**657 claims**).
- Health insurance recorded the highest individual claim amount (**961,424.75**).
- Outlier percentages ranged from **5.94% to 6.68%** across insurance types.
- **Decision:** Retain all outliers because unusually large claims may represent legitimate high-severity insurance events. Statistical outliers are not automatically fraudulent or invalid.


In [32]:
# Step 2.7.18 — Detect Payment Amount Outliers by Insurance Type

# Join payments with claim and policy information
payment_outlier_audit = (
    datasets["Payments"]
    .merge(
        datasets["Claims"][["Claim_ID", "Policy_ID", "Claim_Status"]],
        on="Claim_ID",
        how="left",
        validate="many_to_one"
    )
    .merge(
        datasets["Policies"][["Policy_ID", "Policy_Type"]],
        on="Policy_ID",
        how="left",
        validate="many_to_one"
    )
)

# Calculate IQR boundaries for each insurance type
payment_bounds = (
    payment_outlier_audit
    .groupby("Policy_Type")["Payment_Amount"]
    .agg(
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

payment_bounds["IQR"] = (
    payment_bounds["Q3"] - payment_bounds["Q1"]
)

payment_bounds["Lower_Bound"] = (
    payment_bounds["Q1"] - 1.5 * payment_bounds["IQR"]
)

payment_bounds["Upper_Bound"] = (
    payment_bounds["Q3"] + 1.5 * payment_bounds["IQR"]
)

# Attach the relevant boundaries to each payment
payment_outlier_audit = payment_outlier_audit.merge(
    payment_bounds[
        ["Policy_Type", "Lower_Bound", "Upper_Bound"]
    ],
    on="Policy_Type",
    how="left",
    validate="many_to_one"
)

# Flag statistical outliers
payment_outlier_audit["Is_Outlier"] = (
    (payment_outlier_audit["Payment_Amount"] <
     payment_outlier_audit["Lower_Bound"]) |
    (payment_outlier_audit["Payment_Amount"] >
     payment_outlier_audit["Upper_Bound"])
)

# Summarize results by insurance type
payment_outlier_summary = (
    payment_outlier_audit
    .groupby("Policy_Type")
    .agg(
        Total_Payments=("Payment_ID", "count"),
        Average_Payment=("Payment_Amount", "mean"),
        Maximum_Payment=("Payment_Amount", "max"),
        Outlier_Count=("Is_Outlier", "sum")
    )
    .reset_index()
)

payment_outlier_summary["Outlier_Percentage"] = (
    payment_outlier_summary["Outlier_Count"]
    / payment_outlier_summary["Total_Payments"] * 100
)

print("Payment Amount Outlier Summary:")
display(payment_outlier_summary.round(2))

print("\nIQR Boundaries by Insurance Type:")
display(payment_bounds.round(2))

print("\nTotal Outlier Payments:")
print(int(payment_outlier_audit["Is_Outlier"].sum()))

print("\nSample Outlier Payments:")
display(
    payment_outlier_audit.loc[
        payment_outlier_audit["Is_Outlier"],
        [
            "Payment_ID",
            "Claim_ID",
            "Policy_Type",
            "Payment_Amount",
            "Claim_Status"
        ]
    ].head(10)
)

Payment Amount Outlier Summary:


,Policy_Type,Total_Payments,Average_Payment,Maximum_Payment,Outlier_Count,Outlier_Percentage
0,Auto,6079,32691.74,448852.28,411,6.76
1,Health,4843,33288.05,634466.82,312,6.44
2,Home,2786,32248.11,418304.38,189,6.78
3,Life,1833,30816.07,366497.30,116,6.33



IQR Boundaries by Insurance Type:


,Policy_Type,Q1,Q3,IQR,Lower_Bound,Upper_Bound
0,Auto,12913.52,40559.27,27645.75,-28555.11,82027.90
1,Health,12841.74,40959.63,28117.89,-29335.10,83136.47
2,Home,12386.96,40138.23,27751.27,-29239.94,81765.13
3,Life,12280.82,38334.77,26053.95,-26800.10,77415.69



Total Outlier Payments:
1028

Sample Outlier Payments:


,Payment_ID,Claim_ID,Policy_Type,Payment_Amount,Claim_Status
48,49,8877,Auto,448852.28,Settled
125,126,4043,Auto,84205.69,Approved
165,166,5933,Health,222535.41,Settled
182,183,24910,Auto,97863.50,Approved
225,226,7783,Health,128712.81,Approved
232,233,12375,Life,120034.94,Approved
233,234,6795,Auto,177703.00,Approved
237,238,20391,Home,95050.97,Settled
283,284,16670,Health,184937.45,Settled
322,323,3153,Home,118025.94,Settled


### Payment Amount Outlier Analysis — Key Findings

- **1,028 payments (6.61%)** were identified as statistical outliers using the IQR method separately for each insurance type.
- Home insurance recorded the highest outlier percentage (**6.78%**), while Auto recorded the highest outlier count (**411**).
- Health insurance recorded the highest individual payment amount (**634,466.82**).
- Earlier reconciliation confirmed that no payment exceeds its associated claim amount.
- **Decision:** Retain all payment outliers for claim severity and settlement analysis. Statistical outliers do not automatically indicate errors or fraud.

In [33]:
# Step 2.7.19 — Validate Payment Method Categories and Distribution

payment_method_audit = datasets["Payments"].copy()

# Examine original payment method values
print("Payment Method Distribution:")

payment_method_summary = (
    payment_method_audit["Payment_Method"]
    .value_counts(dropna=False)
    .rename_axis("Payment_Method")
    .reset_index(name="Payment_Count")
)

payment_method_summary["Percentage"] = (
    payment_method_summary["Payment_Count"]
    / len(payment_method_audit) * 100
).round(2)

display(payment_method_summary)

# Check missing and blank payment methods
missing_methods = (
    payment_method_audit["Payment_Method"].isna().sum()
)

blank_methods = (
    payment_method_audit["Payment_Method"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

# Check inconsistent formatting
original_methods = (
    payment_method_audit["Payment_Method"]
    .dropna()
    .astype(str)
    .unique()
)

normalized_methods = (
    payment_method_audit["Payment_Method"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
)

# Create validation summary
method_validation = pd.DataFrame({
    "Validation": [
        "Missing Payment Methods",
        "Blank Payment Methods",
        "Unique Original Categories",
        "Unique Normalized Categories"
    ],
    "Result": [
        int(missing_methods),
        int(blank_methods),
        len(original_methods),
        len(normalized_methods)
    ]
})

print("\nPayment Method Validation Summary:")
display(method_validation)

print("\nOriginal Payment Method Categories:")
print(sorted(original_methods))

Payment Method Distribution:


,Payment_Method,Payment_Count,Percentage
0,Bank Transfer,11153,71.77
1,UPI,2524,16.24
2,Cheque,1864,11.99



Payment Method Validation Summary:


,Validation,Result
0,Missing Payment Methods,0
1,Blank Payment Methods,0
2,Unique Original Categories,3
3,Unique Normalized Categories,3



Original Payment Method Categories:
['Bank Transfer', 'Cheque', 'UPI']


In [34]:
# Step 2.7.20 — Validate Payment Dates Against Analysis Cutoff

payment_date_audit = datasets["Payments"].copy()

# Convert payment dates to datetime
payment_date_audit["Payment_Date"] = pd.to_datetime(
    payment_date_audit["Payment_Date"],
    errors="coerce"
)

# Identify payments after the analysis cutoff
future_payment_mask = (
    payment_date_audit["Payment_Date"] > reference_date
)

future_payments = payment_date_audit.loc[
    future_payment_mask
].copy()

# Calculate summary metrics
total_payments = len(payment_date_audit)
future_payment_count = len(future_payments)

future_payment_percentage = (
    future_payment_count / total_payments * 100
)

print("Analysis Cutoff:", reference_date.date())
print("Total Payment Records:", total_payments)
print("Payments After Cutoff:", future_payment_count)
print(
    "Percentage After Cutoff:",
    round(future_payment_percentage, 2),
    "%"
)

print("\nFuture-Dated Payment Amount Summary:")
display(
    future_payments["Payment_Amount"]
    .describe()
    .round(2)
)

print("\nPayments After Cutoff by Month:")
display(
    future_payments
    .assign(
        Payment_Month=future_payments[
            "Payment_Date"
        ].dt.to_period("M")
    )
    .groupby("Payment_Month")
    .agg(
        Payment_Count=("Payment_ID", "count"),
        Total_Amount=("Payment_Amount", "sum")
    )
    .round(2)
)

print("\nSample Payments After Cutoff:")
display(
    future_payments[
        [
            "Payment_ID",
            "Claim_ID",
            "Payment_Date",
            "Payment_Amount",
            "Payment_Method"
        ]
    ].head(10)
)

Analysis Cutoff: 2026-09-30
Total Payment Records: 15541
Payments After Cutoff: 189
Percentage After Cutoff: 1.22 %

Future-Dated Payment Amount Summary:


count       189.00
mean      32475.87
std       45796.17
min        2553.25
25%       12769.16
50%       23139.56
75%       41285.05
max      572755.54
Name: Payment_Amount, dtype: float64


Payments After Cutoff by Month:


,Payment_Count,Total_Amount
Payment_Month,,
2026-10,98,3616289.62
2026-11,76,2059165.27
2026-12,15,462484.44



Sample Payments After Cutoff:


,Payment_ID,Claim_ID,Payment_Date,Payment_Amount,Payment_Method
12,13,9083,2026-11-08,27973.17,Bank Transfer
130,131,13122,2026-10-29,49416.01,Bank Transfer
169,170,5558,2026-11-01,13869.63,Bank Transfer
244,245,18017,2026-12-06,48790.27,Bank Transfer
337,338,22318,2026-10-09,12171.39,Bank Transfer
387,388,2327,2026-10-12,3568.48,Bank Transfer
441,442,6359,2026-11-16,11979.24,Bank Transfer
442,443,16334,2026-10-19,12769.16,Bank Transfer
720,721,6968,2026-12-03,19193.92,Cheque
724,725,11584,2026-10-17,12785.84,Bank Transfer


### Payment Date Cutoff — Key Findings

- The analysis reference date is **September 30, 2026**.
- **189 payments (1.22%)**, totaling **6,137,939.33**, were recorded after the reference date.
- These payments occur between October and December 2026.
- Including these records in historical payment KPIs would overstate payments as of the cutoff date.
- **Decision:** Retain all original payment records and apply date-based filtering when calculating historical settlement and financial KPIs.

In [35]:
# Step 2.7.21 — Analyze Claim Amount and Payment Amount Gaps

# Use the existing claim-level payment reconciliation
claim_gap_audit = payment_reconciliation.copy()

# Calculate payment gap for each claim
claim_gap_audit["Payment_Gap"] = (
    claim_gap_audit["Claim_Amount"]
    - claim_gap_audit["Total_Payment"]
)

# Calculate payment-to-claim ratio
claim_gap_audit["Payment_Ratio"] = np.where(
    claim_gap_audit["Claim_Amount"] > 0,
    claim_gap_audit["Total_Payment"]
    / claim_gap_audit["Claim_Amount"],
    np.nan
)

# Summarize claims by payment availability
print("Claim Payment Availability Summary:")

display(
    claim_gap_audit.groupby("Claim_Status").agg(
        Total_Claims=("Claim_ID", "count"),
        Claims_With_Payment=(
            "Payment_Count",
            lambda x: (x > 0).sum()
        ),
        Claims_Without_Payment=(
            "Payment_Count",
            lambda x: (x == 0).sum()
        )
    )
)

# Examine payment ratios only for claims with payments
paid_claims = claim_gap_audit[
    claim_gap_audit["Payment_Count"] > 0
]

print("\nPayment Ratio Statistics for Paid Claims:")

display(
    paid_claims["Payment_Ratio"]
    .describe()
    .round(4)
)

# Summarize payment gaps by claim status
print("\nClaim-Payment Gap by Claim Status:")

display(
    claim_gap_audit.groupby("Claim_Status").agg(
        Total_Claim_Amount=("Claim_Amount", "sum"),
        Total_Payment_Amount=("Total_Payment", "sum"),
        Total_Payment_Gap=("Payment_Gap", "sum")
    ).round(2)
)

print("\nOverall Financial Summary:")

print(
    "Total Claim Amount:",
    round(claim_gap_audit["Claim_Amount"].sum(), 2)
)

print(
    "Total Recorded Payment Amount:",
    round(claim_gap_audit["Total_Payment"].sum(), 2)
)

print(
    "Total Claim-Payment Gap:",
    round(claim_gap_audit["Payment_Gap"].sum(), 2)
)

Claim Payment Availability Summary:


,Total_Claims,Claims_With_Payment,Claims_Without_Payment
Claim_Status,,,
Approved,6100,5704,396
Pending,4075,0,4075
Rejected,4256,0,4256
Settled,10569,9837,732



Payment Ratio Statistics for Paid Claims:


count    15541.0000
mean         0.7757
std          0.1291
min          0.5500
25%          0.6643
50%          0.7764
75%          0.8884
max          0.9999
Name: Payment_Ratio, dtype: float64


Claim-Payment Gap by Claim Status:


,Total_Claim_Amount,Total_Payment_Amount,Total_Payment_Gap
Claim_Status,,,
Approved,2.524593e+08,1.827470e+08,6.971225e+07
Pending,1.713947e+08,0.000000e+00,1.713947e+08
Rejected,1.820868e+08,0.000000e+00,1.820868e+08
Settled,4.477088e+08,3.235292e+08,1.241796e+08



Overall Financial Summary:
Total Claim Amount: 1053649601.45
Total Recorded Payment Amount: 506276245.77
Total Claim-Payment Gap: 547373355.68


In [36]:
# Step 2.7.22 — Consolidate Final Data Quality Audit Findings

audit_findings = [
    {
        "Audit_Area": "Missing Values",
        "Affected_Records": 0,
        "Status": "PASS",
        "Recommended_Action": "No action required"
    },
    {
        "Audit_Area": "Duplicate Records",
        "Affected_Records": 0,
        "Status": "PASS",
        "Recommended_Action": "No action required"
    },
    {
        "Audit_Area": "Foreign Key Integrity",
        "Affected_Records": 0,
        "Status": "PASS",
        "Recommended_Action": "No action required"
    },
    {
        "Audit_Area": "Policy Start Before Customer Join",
        "Affected_Records": 3512,
        "Status": "REVIEW",
        "Recommended_Action": "Verify date definitions; retain originals"
    },
    {
        "Audit_Area": "Policy Status vs Coverage Dates",
        "Affected_Records": 7924,
        "Status": "REVIEW",
        "Recommended_Action": "Create date-derived status for reporting"
    },
    {
        "Audit_Area": "Claims Exceeding Coverage",
        "Affected_Records": 143,
        "Status": "REVIEW",
        "Recommended_Action": "Flag for financial analysis"
    },
    {
        "Audit_Area": "Payments Exceeding Coverage",
        "Affected_Records": 34,
        "Status": "REVIEW",
        "Recommended_Action": "Flag for financial analysis"
    },
    {
        "Audit_Area": "Settled Claims Without Payments",
        "Affected_Records": 732,
        "Status": "REVIEW",
        "Recommended_Action": "Flag payment-status inconsistency"
    },
    {
        "Audit_Area": "Claim Type vs Policy Type",
        "Affected_Records": 0,
        "Status": "PASS",
        "Recommended_Action": "No action required"
    },
    {
        "Audit_Area": "Risk Category Classification",
        "Affected_Records": 0,
        "Status": "PASS",
        "Recommended_Action": "No action required"
    },
    {
        "Audit_Area": "Invalid Fraud Flags",
        "Affected_Records": 0,
        "Status": "PASS",
        "Recommended_Action": "No action required"
    },
    {
        "Audit_Area": "Payments Exceeding Claim Amount",
        "Affected_Records": 0,
        "Status": "PASS",
        "Recommended_Action": "No action required"
    },
    {
        "Audit_Area": "Auto Policies Without Vehicle Records",
        "Affected_Records": 2389,
        "Status": "REVIEW",
        "Recommended_Action": "Document vehicle coverage limitation"
    },
    {
        "Audit_Area": "Cross-Region Agent Assignments",
        "Affected_Records": 11976,
        "Status": "REVIEW",
        "Recommended_Action": "Retain; verify assignment rules"
    },
    {
        "Audit_Area": "Premium Ratio Outliers",
        "Affected_Records": 380,
        "Status": "REVIEW",
        "Recommended_Action": "Retain statistical outliers"
    },
    {
        "Audit_Area": "Claim Amount Outliers",
        "Affected_Records": 1585,
        "Status": "REVIEW",
        "Recommended_Action": "Retain for severity analysis"
    },
    {
        "Audit_Area": "Payment Amount Outliers",
        "Affected_Records": 1028,
        "Status": "REVIEW",
        "Recommended_Action": "Retain for settlement analysis"
    },
    {
        "Audit_Area": "Payments After Analysis Cutoff",
        "Affected_Records": 189,
        "Status": "REVIEW",
        "Recommended_Action": "Apply cutoff filter to historical KPIs"
    },
    {
        "Audit_Area": "Payment Method Categories",
        "Affected_Records": 0,
        "Status": "PASS",
        "Recommended_Action": "No action required"
    },
    {
        "Audit_Area": "Policy-Agent Assignment",
        "Affected_Records": 0,
        "Status": "PASS",
        "Recommended_Action": "No action required"
    }
]

# Create final audit summary
final_audit_summary = pd.DataFrame(audit_findings)

print("FINAL DATA QUALITY AUDIT SUMMARY")
display(final_audit_summary)

print("\nAudit Status Distribution:")
display(
    final_audit_summary["Status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Audit_Checks")
)

print("\nTotal Audit Checks:", len(final_audit_summary))
print(
    "Checks Passed:",
    (final_audit_summary["Status"] == "PASS").sum()
)
print(
    "Checks Requiring Review:",
    (final_audit_summary["Status"] == "REVIEW").sum()
)

FINAL DATA QUALITY AUDIT SUMMARY


,Audit_Area,Affected_Records,Status,Recommended_Action
0,Missing Values,0,PASS,No action required
1,Duplicate Records,0,PASS,No action required
2,Foreign Key Integrity,0,PASS,No action required
3,Policy Start Before Customer Join,3512,REVIEW,Verify date definitions; retain originals
4,Policy Status vs Coverage Dates,7924,REVIEW,Create date-derived status for reporting
5,Claims Exceeding Coverage,143,REVIEW,Flag for financial analysis
6,Payments Exceeding Coverage,34,REVIEW,Flag for financial analysis
7,Settled Claims Without Payments,732,REVIEW,Flag payment-status inconsistency
8,Claim Type vs Policy Type,0,PASS,No action required
9,Risk Category Classification,0,PASS,No action required



Audit Status Distribution:


,Status,Audit_Checks
0,REVIEW,11
1,PASS,9



Total Audit Checks: 20
Checks Passed: 9
Checks Requiring Review: 11



### Key Findings

- Evaluated eight insurance datasets through 20 consolidated audit checks: **9 PASS and 11 REVIEW**.
- No missing values, duplicate records, or foreign-key violations were identified.
- Identified 3,512 policies starting before customer joining dates and 7,924 policy-status discrepancies.
- Found 143 claims and 34 payments exceeding associated policy coverage.
- Identified 732 settled claims without payment records.
- Detected 1,585 claim amount outliers and 1,028 payment amount outliers.
- Identified 189 payments occurring after the September 30, 2026 analysis cutoff.

### Cleaning Decisions

- Preserve original records and avoid unnecessary deletion.
- Standardize data types and create analytical flags where needed.
- Retain statistical outliers for risk and claim severity analysis.
- Maintain original policy statuses while creating date-derived coverage statuses.
- Apply appropriate date filters to historical financial KPIs.

